In [ ]:
import numpy as np
import utils
import matplotlib.pyplot as plt
from os.path import exists
import json
plt.style.use('/home/adiaz/Plots/thesis.mplstyle')

In [ ]:
flux_name = "AIRS_mceq121_2011_sib_HG_E3"
crgamma = "0.25"
dir_base = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fits/FitToData/" + flux_name + "/crSlopeWidth_" + crgamma+ "/FS_0.50/"

In [ ]:
f =open(dir_base + "BestFitBlind.json")
BF=json.load(f)
f.close()

In [ ]:
minimum_llhs = np.load(dir_base + "/llhs.npy")
point_string = BF['Point']

In [ ]:
llh_min = minimum_llhs[:,:,3].min()

In [ ]:
data=2000*[None]
llhs = np.zeros(2000)
for i in range(2000):
    dirname0 = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fits/FitToData/"+flux_name+ "/crSlopeWidth_"+ crgamma +"/BlindFit/n_seeds_-1/" 
    dirname1 = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fits/FitToData/"+flux_name+ "/crSlopeWidth_"+ crgamma +"/BlindFit/n_seeds_4/" 
    filename = "{point_string}_Realization_{i}.json".format(point_string = point_string, i = i)
    
    f0 =open(dirname0 + filename)
    f1 = open(dirname1 + filename)
    data0=json.load(f0)
    data1=json.load(f1)
    f0.close()
    f1.close()
    llh0 = data0['LLH']
    llh1 = data1['LLH']
    
    if (llh0<llh1):        
        data[i] = data0        
    else:
        data[i] = data1
    
with open('data.json', 'w') as f:
    json.dump(data, f)    

In [ ]:
f = open('data.json')
d = json.load(f)
f.close()

In [ ]:
llhs = np.array([r['LLH'] for r in d])

In [ ]:
llh_min

In [ ]:
fig, ax = plt.subplots()

llh_min_edge = int(np.floor(2*llhs[np.nonzero(llhs)].min()))
llh_max_edge = int(np.ceil(2*llhs[np.nonzero(llhs)].max()))
bins = np.arange(llh_min_edge, llh_max_edge, 1)
n, bin_edges, *_ = ax.hist(2*llhs[np.nonzero(llhs)], cumulative=False, density=True, bins = bins, label = "Ensemble")
bin_centers = (bin_edges[1:]+bin_edges[:-1])/2.
# plt.plot(bin_centers, np.cumsum(n))
ax.vlines(2*llh_min, 0,1.1*n.max(), label = "Observed")
ax.set_title("Ensemble Test")
ax.set_xlabel("-2 LLH")
ax.set_ylabel("PDF")
# ax.axvspan(llh_min, bin_edges.max(), ymax = 0.5, alpha=0.3, color='red')
ax.text(2*2920, 0.015,"p={:.4}".format(((llhs > llh_min).sum())/len(llhs)), size = 15)
ax.legend()
plt.tight_layout()

# plt.savefig("BlindFitLLH.png", dpi = 144)


plt.show()

In [ ]:
BF_file = open(dir_base + "/BestFitBlind.json",'r')
BF_data = json.load(BF_file)
BF_file.close()

In [ ]:
from scipy.stats import poisson
def p_llh(x,mean):
#     return poisson.pmf(x, mean)
    return np.log(poisson.pmf(x, mean))

In [ ]:
cols, rows = np.array(d[0]['Fit']).shape
# fig, axs = plt.subplots(rows, cols, figsize = (20,13))
expectations = np.zeros((cols, rows, 2000))
observed = np.zeros((cols, rows, 2000))
llhs = np.zeros((cols, rows, 2000))
Observed_llh = np.zeros((cols, rows))

In [ ]:
for row in range(rows):
    for col in range(cols):
        print(col, row)
        expectations[col,row] = np.array([r['Fit'][col][row] for r in d])
        observed[col,row] = np.array([r['Realization'][col][row]for r in d]).astype(int)

        llhs[col,row] = [p_llh(o, e) for o,e in zip(observed[col,row], expectations[col,row])]

        Observed_llh[col,row] = p_llh(np.array(BF_data['Realization']).astype(int)[col][row], \
                             np.array(BF_data['Fit'])[col][row])
#         axs[rows-row-1][col].hist(llhs, bins = 100, cumulative=False, density=True)
#         plt.vlines(Observed_llh, 0, 5)
#         p_values[i] = (llhs<Observed_llh).sum()/10000
    #     print(expectations, observed)



In [ ]:
p_values = np.zeros((rows, cols))
for row in range(rows):
    for col in range(cols):
        p_values[row,col] = (llhs[col,row]<Observed_llh[col,row]).sum()/2000



In [ ]:
fig, axs = plt.subplots(rows, cols, figsize = (6*cols,4*rows))
p_values = np.zeros((rows, cols))
for row in range(rows):
    for col in range(cols):
        print(col, row)
        counts, *_ = axs[rows-row-1][col].hist(-2* llhs[col,row], bins = 100, cumulative=False, density=True)
        axs[rows-row-1][col].vlines(-2*Observed_llh[col,row], 0, 1.1*counts.max())
        axs[rows-row-1][col].set_xlabel("-2 LLH")
        axs[rows-row-1][col].set_ylabel("PDF")

plt.tight_layout()
# plt.savefig("binwiseLLH.png", dpi = 144)
plt.show()

In [ ]:
expectations.shape

In [ ]:
np.array(d[0]['Fit']).shape
# plt.pcolormesh(np.array(d[0]['Fit']).T)

In [ ]:
rows

In [ ]:
p_values = np.zeros(260)
for i in range(260):
    print(i)
    expectations = np.array([np.array(r['Fit']).flatten()[i] for r in d])
    observed = np.array([np.array(r['Realization']).flatten()[i] for r in d]).astype(int)
    
    llhs = [p_llh(o, e) for o,e in zip(observed, expectations)]
    
    Observed_llh = p_llh(np.array(BF_data['Realization']).astype(int).flatten()[i], \
                         np.array(BF_data['Fit']).flatten()[i])
#     plt.hist(llhs, bins = 100, cumulative=False, density=True)
#     plt.vlines(Observed_llh, 0, 5)
#     plt.show()
    p_values[i] = (llhs<Observed_llh).sum()/2000
#     print(expectations, observed)
    

In [ ]:
plt.hist(p_values)

In [ ]:
def get_chi2(r):
    cz_data = np.array(r['Realization']).sum(axis=1)
    cz_fit = np.array(r['Fit']).sum(axis=1)

    E_data = np.array(r['Realization']).sum(axis=0)
    E_fit = np.array(r['Fit']).sum(axis=0)

    cz_chi2 = ((cz_data - cz_fit)/np.sqrt(cz_fit))**2


    E_chi2 = ((E_data - E_fit)/np.sqrt(E_fit))**2

    
    return cz_chi2.sum(), E_chi2.sum()

cz_chi2s = np.zeros(2000)
E_chi2s = np.zeros(2000)
for i in range(len(d)):
    
    r = d[i]
    cz_chi2s[i], E_chi2s[i] = get_chi2(r)


In [ ]:
from scipy.stats import chi2
from matplotlib.ticker import MultipleLocator

cz_data = np.array(BF_data['Realization']).sum(axis=1)
cz_fit = np.array(BF_data['Fit']).sum(axis=1)

E_data = np.array(BF_data['Realization']).sum(axis=0)
E_fit = np.array(BF_data['Fit']).sum(axis=0)

# cz_chi2 = ((cz_data - cz_fit)/np.sqrt(cz_fit))**2
# p_cz = 1.-chi2.cdf(cz_chi2.sum(), len(cz_chi2))

# E_chi2 = ((E_data - E_fit)/np.sqrt(E_fit))**2
# p_E = 1.-chi2.cdf(E_chi2.sum(), len(E_chi2))


BF_cz_chi2, BF_E_chi2 = get_chi2(BF_data)
n, *_ = np.histogram(cz_chi2s, bins = 100)
p_cz = (cz_chi2s>BF_cz_chi2).sum()/len(cz_chi2s)
p_E = (E_chi2s>BF_E_chi2).sum()/len(E_chi2s)


fig, axs = plt.subplots(2,1,sharex=True, gridspec_kw={'height_ratios': [2, 1]})

fig.suptitle(r"Zenith Distribution", fontsize = 15)
axs[0].step(BF_data['Z_Edges'],[cz_data[0]] + cz_data.tolist(), label = 'Data')
axs[0].step(BF_data['Z_Edges'],[cz_fit[0]] + cz_fit.tolist(), label = 'Best Fit')
axs[0].set_ylabel("Events")
axs[0].legend()

axs[1].step(BF_data['Z_Edges'], [(cz_data/cz_fit)[0]] + (cz_data/cz_fit).tolist())
axs[1].errorbar(BF_data['Z_centers'], cz_data/cz_fit, yerr=np.sqrt(cz_data)/cz_fit, fmt='k.')
axs[1].set_xlabel(r"$\cos(\theta)$")
axs[1].set_ylabel("Ratios")
axs[1].set_ylim(0.95, 1.05)

axs[1].xaxis.set_minor_locator(MultipleLocator(0.1))

axs[1].grid(alpha = 0.5, which = 'both')


axs[0].text(-0.3, 10000,"p={:.4}".format(p_cz), size = 15)
plt.tight_layout()
fig.subplots_adjust(hspace=0)
plt.subplots_adjust(top=0.9)


# plt.savefig("1DZenith.png", dpi = 144)


plt.show()

fig, axs = plt.subplots(2,1,sharex=True, gridspec_kw={'height_ratios': [2, 1]})
axs[0].set_xscale('log')
axs[0].set_yscale('log')
axs[0].step(10**np.array(BF_data['E_Edges']),[E_data[0]] + E_data.tolist(), label='Data')
axs[0].step(10**np.array(BF_data['E_Edges']),[E_fit[0]] + E_fit.tolist(), label='Best Fit' )
axs[0].set_ylabel("Events")
axs[0].legend()

axs[1].set_xscale('log')
axs[1].step(10**np.array(BF_data['E_Edges']), [(E_data/E_fit)[0]] + (E_data/E_fit).tolist())
axs[1].errorbar(10**np.array(BF_data['E_centers']), E_data/E_fit, yerr = np.sqrt(E_data)/E_fit, fmt = 'k.')
axs[1].set_xlabel(r"Energy [GeV]")
axs[1].set_ylim(0.90, 1.10)
axs[1].set_ylabel("Ratios")

axs[1].grid(alpha = 0.5, which = 'both')

axs[0].text(500, 1000,"p={:.4}".format(p_E), size = 15)

plt.tight_layout()
fig.subplots_adjust(hspace=0)
plt.subplots_adjust(top=0.9)


fig.suptitle("Energy Distribution", fontsize = 15)



# plt.savefig("1DEnergy.png", dpi = 144)
plt.show()

In [ ]:
BF_cz_chi2, BF_E_chi2 = get_chi2(BF_data)

n, *_ = plt.hist(cz_chi2s, bins = 100)
plt.vlines(BF_cz_chi2, 0, n.max())
print((cz_chi2s>BF_cz_chi2).sum()/10000.)
plt.show()

n, *_ = plt.hist(E_chi2s, bins = 100)
plt.vlines(BF_E_chi2, 0, n.max())
print((E_chi2s>BF_E_chi2).sum()/10000.)
plt.show()

In [ ]:
E_chi2 = ((E_data - E_fit)/np.sqrt(E_fit))**2
print((E_data - E_fit)/np.sqrt(E_fit)>3)
print(1.-chi2.cdf(E_chi2.sum(), len(E_chi2)))

In [ ]:
BF_data

In [ ]:
# from matplotlib importrcParams
plt.rcParams['figure.dpi']

In [ ]:
llh_min